In [54]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import datetime as dt

In [56]:
# Parameters
min_rest_days = 3
max_matches_per_venue_day = 1
min_matches_per_city = 3
max_matches_per_city = 6
max_matches_per_kickoff = 20

# Weights for fan heat exposure indoor and outdoor
# 80% time spent indoors / 20% time spent outdoors
w_in = 0.8
w_out = 0.2

# FIFA matchday windows
matchday_dates = {
    1: [
        dt.date(2026, 6, 11),
        dt.date(2026, 6, 12),
        dt.date(2026, 6, 13),
        dt.date(2026, 6, 14),
        dt.date(2026, 6, 15),
        dt.date(2026, 6, 16)
    ],
    2: [
        dt.date(2026, 6, 17),
        dt.date(2026, 6, 18),
        dt.date(2026, 6, 19),
        dt.date(2026, 6, 20),
        dt.date(2026, 6, 21),
        dt.date(2026, 6, 22)
    ],
    3: [
        dt.date(2026, 6, 23),
        dt.date(2026, 6, 24),
        dt.date(2026, 6, 25),
        dt.date(2026, 6, 26),
        dt.date(2026, 6, 27)
    ]
}

matches = pd.read_csv("group_positions.csv")
venues = pd.read_csv("stadiums.csv")
wbgt = pd.read_csv("WBGT_data.csv")

# Convert CSV dates from strings to Python date objects
wbgt["Date"] = pd.to_datetime(wbgt["Date"]).dt.date

In [57]:
# Map kickoff times to CSV columns
kickoff_columns = {
    "12:00": "WBGT_12",
    "15:00": "WBGT_15",
    "18:00": "WBGT_18",
    "21:00": "WBGT_21"
}

# All possible schedule options
options_by_matchday = {}
option_id = 0

for md in matchday_dates:

    options = []

    for day in matchday_dates[md]:

        for _, venue in venues.iterrows():

            # Find weather for this stadium based on its city on this date
            city_weather = wbgt[
                (wbgt["City"] == venue.city) &
                (wbgt["Date"] == day)
            ]

            weather_row = city_weather.iloc[0]

            for kickoff_time, wbgt_column in kickoff_columns.items():

                # Outdoor WBGT from CSV
                outdoor_wbgt = weather_row[wbgt_column]

                # Indoor WBGT (21 if climate controlled, otherwise same as outdoor)
                if weather_row["Climate_Controlled"] == 1:
                    indoor_wbgt = 21.0
                else:
                    indoor_wbgt = outdoor_wbgt

                # Weighted fan heat exposure
                risk = (
                    w_in * indoor_wbgt +
                    w_out * outdoor_wbgt
                )

                options.append({
                    "option_id": option_id,
                    "matchday": md,
                    "city": venue.city,
                    "stadium": venue.stadium_name,
                    "capacity": venue.capacity,
                    "date": day,
                    "date_idx": (day - dt.date(2026, 1, 1)).days,
                    "kickoff_time": kickoff_time,
                    "wbgt_outdoor": outdoor_wbgt,
                    "wbgt_indoor": indoor_wbgt,
                    "risk": round(risk, 2),
                    "climate_controlled": weather_row["Climate_Controlled"]
                })

                option_id += 1

    options_by_matchday[md] = pd.DataFrame(options)


# Combine all schedule options
schedule_options = (
    pd.concat(options_by_matchday.values(), ignore_index=True)
    .set_index("option_id")
)

print(f"Total schedule options = {len(schedule_options)}")

schedule_options.head()

Total schedule options = 1088


,matchday,city,stadium,capacity,date,date_idx,kickoff_time,wbgt_outdoor,wbgt_indoor,risk,climate_controlled
option_id,,,,,,,,,,,
0,1,Kansas City,Arrowhead Stadium,69045,2026-06-11,161,12:00,25.84,25.84,25.84,0
1,1,Kansas City,Arrowhead Stadium,69045,2026-06-11,161,15:00,23.83,23.83,23.83,0
2,1,Kansas City,Arrowhead Stadium,69045,2026-06-11,161,18:00,24.62,24.62,24.62,0
3,1,Kansas City,Arrowhead Stadium,69045,2026-06-11,161,21:00,22.53,22.53,22.53,0
4,1,Toronto,BMO Field,43036,2026-06-11,161,12:00,21.98,21.98,21.98,0


In [58]:
m = gp.Model("wbgt_heat_risk_scheduling")

# Decision variable - x[match_id, option_id]
x = {}

# Stores which schedule options are available for each match
options_for_match = {}

for _, match in matches.iterrows():
    # A match can only use schedule options from its assigned matchday
    valid_options = list(
        schedule_options[
            schedule_options.matchday == match.matchday
        ].index
    )

    options_for_match[match.match_id] = valid_options

    for option_id in valid_options:
        x[match.match_id, option_id] = m.addVar(
            vtype=GRB.BINARY,
        )

print(f"Decision variables: {len(x)}")

Decision variables: 26112


In [59]:
# Minimize heat-risk exposure per person
obj = gp.quicksum(
    schedule_options.loc[option_id, "risk"] * schedule_options.loc[option_id, "capacity"]* x[match_id, option_id]

    for match_id, options in options_for_match.items()
    for option_id in options
)

m.setObjective(obj, GRB.MINIMIZE)

In [61]:
# Constraint - every match scheduled exactly once
for match_id, options in options_for_match.items():
    m.addConstr(gp.quicksum(x[match_id, option_id] for option_id in options) == 1)

# Maximum 1 match at each stadium per day
for (stadium, date), option_ids in schedule_options.groupby(
    ["stadium", "date"]
).groups.items():

    m.addConstr(
        gp.quicksum(
            x[match_id, option_id]
            for match_id, options in options_for_match.items()
            for option_id in options
            if option_id in option_ids
        ) <= max_matches_per_venue_day
    )

# Minimum and maximum matches per city
for city in venues.city.unique():

    city_options = [
        x[match_id, option_id]
        for match_id, options in options_for_match.items()
        for option_id in options
        if schedule_options.loc[option_id, "city"] == city
    ]

    m.addConstr(gp.quicksum(city_options) >= min_matches_per_city)
    m.addConstr(gp.quicksum(city_options) <= max_matches_per_city)


# Get the selected date for a match
def get_match_date(match_id):

    date = gp.LinExpr()

    for option_id in options_for_match[match_id]:
        date += (
            schedule_options.loc[option_id, "date_idx"]
            * x[match_id, option_id]
        )

    return date

# Teams must have at least 3 days between matches
for group, group_matches in matches.groupby("group"):

    for team in [1, 2, 3, 4]:

        team_matches = []

        for matchday in [1, 2, 3]:

            matchday_matches = group_matches[
                group_matches.matchday == matchday
            ]

            team_match = matchday_matches[
                (matchday_matches.team1 == team) |
                (matchday_matches.team2 == team)
            ]

            if len(team_match) == 1:
                team_matches.append(team_match.iloc[0].match_id)

        for previous_match, next_match in zip(
            team_matches,
            team_matches[1:]
        ):

            m.addConstr(
                get_match_date(next_match)
                - get_match_date(previous_match)
                >= min_rest_days
            )

# Maximum number of matches at each kickoff time
for kickoff in schedule_options.kickoff_time.unique():

    kickoff_options = [
        x[match_id, option_id]
        for match_id, options in options_for_match.items()
        for option_id in options
        if schedule_options.loc[option_id, "kickoff_time"] == kickoff
    ]

    m.addConstr(
        gp.quicksum(kickoff_options) <= max_matches_per_kickoff
    )

m.update()

print(f"Total constraints: {m.NumConstrs}")

Total constraints: 952


In [62]:
m.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i5-1335U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 952 rows, 26112 columns and 402432 nonzeros (Min)
Model fingerprint: 0xb8f09328
Model has 26112 linear objective coefficients
Variable types: 0 continuous, 26112 integer (26112 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [6e+05, 2e+06]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]

Found heuristic solution: objective 9.811788e+07
Presolve added 0 rows and 96 columns
Presolve removed 380 rows and 0 columns
Presolve time: 0.21s
Presolved: 572 rows, 26208 columns, 150816 nonzeros
Variable types: 0 continuous, 26208 integer (26112 binary)

Root relaxation: objective 7.654609e+07, 1527 iterations, 0.06 seconds (0.15 work units)

    Nodes    |    Current Node    |    

In [63]:
results = []

if m.SolCount > 0:
    for match_id, options in options_for_match.items():
        for option_id in options:
            if x[match_id, option_id].X > 0.5:
                option = schedule_options.loc[option_id]
                match = matches[
                    matches.match_id == match_id
                ].iloc[0]

                results.append({
                    "match_id": match_id,
                    "group": match.group,
                    "team1": match.team1,
                    "team2": match.team2,
                    "matchday": match.matchday,

                    "city": option.city,
                    "stadium": option.stadium,
                    "date": option.date,
                    "kickoff_time": option.kickoff_time,

                    "risk": option.risk,
                    "capacity": option.capacity,

                    "person_weighted_risk":
                        option.risk * option.capacity
                })

    results_df = (
        pd.DataFrame(results)
        .sort_values(["group","matchday"])
    )

    print("Total person-weighted risk:", m.ObjVal)
    print("Average WBGT per match:", results_df["risk"].mean())
    results_df

else:

    print("No feasible solution found.")

Total person-weighted risk: 76546088.81
Average WBGT per match: 16.820277777777775


In [64]:
# Save optimized schedule
results_df.to_csv(
    "optimized.csv",
    index=False
)

In [65]:
# Load official FIFA schedule
real_schedule = pd.read_csv("real_schedule.csv")

# Convert dates
real_schedule["date"] = pd.to_datetime(
    real_schedule["date"]
).dt.date

# Round kickoff times to match available WBGT data
def get_wbgt_column(kickoff_time):
    hour = int(kickoff_time.split(":")[0])

    if hour <= 13:
        return "WBGT_12"
    elif hour <= 16:
        return "WBGT_15"
    elif hour <= 19:
        return "WBGT_18"
    else:
        return "WBGT_21"

risks = []

# Calculate WBGT risk for each match in the official schedule
for _, match in real_schedule.iterrows():

    weather = wbgt[
        (wbgt["City"] == match["city"]) &
        (wbgt["Date"] == match["date"])
    ].iloc[0]

    outdoor_wbgt = weather[
        get_wbgt_column(match["kickoff_time_local"])
    ]

    if weather["Climate_Controlled"] == 1:
        indoor_wbgt = 21.0
    else:
        indoor_wbgt = outdoor_wbgt

    risk = (
        w_in * indoor_wbgt +
        w_out * outdoor_wbgt
    )

    risks.append(risk)

real_schedule["risk"] = risks

# Add capacity
real_schedule = real_schedule.merge(
    venues[["city", "capacity"]],
    on="city"
)

# Person-weighted risk
real_schedule["person_weighted_risk"] = (
    real_schedule["risk"] *
    real_schedule["capacity"]
)

# Results
print("Total person-weighted risk:", real_schedule["person_weighted_risk"].sum())
print("Average WBGT per match:", real_schedule["risk"].mean())

Total person-weighted risk: 97400857.07000001
Average WBGT per match: 20.809916666666666
